# Constitutional RAG

## Prerequisits

In [37]:
# imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import pymupdf
import os
import re 

In [42]:
# --- 1. The "Bulletproof" PDF Cleaning Function ---
def clean_pdf_text(text: str) -> str:
    # 1. Replace tabs with spaces
    text = text.replace('\t', ' ')
    
    # 2. Fix words broken by hyphens at line breaks (e.g., "con-\nstitution")
    text = re.sub(r'(\w+)-\n(\w+)', r'\1\2', text)
    
    # 3. Remove isolated page numbers (lines containing ONLY a number)
    text = re.sub(r'^\s*\d{1,4}\s*$', '', text, flags=re.MULTILINE)
    
    # 4. Remove running headers/footers (e.g., "Chapter 1: Founding Provisions")
    text = re.sub(r'(?m)^\s*Chapter \d+:\s+[A-Za-z ]+\s*$', '', text)
    
    # 5. THE MAGIC TRICK: Fix PDF line-wraps without losing paragraph breaks
    # PDFs put a single \n at the end of every visual line. We want to turn those into spaces,
    # but keep the double \n\n that separate actual paragraphs.
    text = text.replace('\n\n', '[[PARA]]')  # Temporarily hide real paragraphs
    text = text.replace('\n', ' ')           # Turn visual line-breaks into spaces
    text = text.replace('[[PARA]]', '\n\n')  # Restore real paragraphs
    
    # 6. Collapse multiple spaces into a single space
    text = re.sub(r' +', ' ', text)
    
    return text.strip()


def clean_toc_index(raw_text):
    # Strip the "Junk" (Table of Contents & Title Page)
    preamble_marker = "PREAMBLE\nWe, the people of South Africa"
    preamble_index = raw_text.find(preamble_marker)

    if preamble_index != -1:
        raw_text = raw_text[preamble_index:]

    # --- NEW: Strip the Index (appears at the end) ---
    # Look for "INDEX" as a standalone heading
    index_marker = "\nINDEX\n"
    index_position = raw_text.find(index_marker)

    if index_position != -1:
        raw_text = raw_text[:index_position]
        print(f"✓ Removed Index starting at position: {index_position}")
    else:
        # Try alternative formats
        alt_markers = ["\nI N D E X\n", "\nINDEX\n\n", "INDEX\n"]
        for marker in alt_markers:
            index_position = raw_text.find(marker)
            if index_position != -1:
                raw_text = raw_text[:index_position]
                print(f"✓ Removed Index using alternative marker")
                break

    return raw_text

In [44]:
# --- 1.  File path
file_path = "../data/RSAConstitution-eng.pdf"


# --- 2. Extract Text using PyMuPDF ---
doc = pymupdf.open(file_path)
raw_text = ""
for page in doc:
    raw_text += page.get_text("text") + "\n"


# --- 3. Strip the "Junk" (Table of Contents & Title Page) ---
# Clean toc and index
clean_text = clean_toc_index(raw_text)

# Clean the text
text = clean_pdf_text(clean_text)


# --- 4. Chapter metadata ---
chapter_pattern = r'CHAPTER\s+\d+\s+[A-Z][A-Z\s\-]*?(?=\s+[A-Z][a-z]|\s+\d+\.)'

# Find all chapter headings and their positions
matches = list(re.finditer(chapter_pattern, text))
print(f"\n✓ Found {len(matches)} chapters")

# Build chapter documents by slicing the text
chapter_documents = []
current_chapter = "PREAMBLE"
current_start = 0

for match in matches:
    # 1. Save the content BEFORE this chapter (belongs to previous chapter)
    if current_start < match.start():
        content = text[current_start:match.start()].strip()
        if content:
            chapter_documents.append(Document(
                page_content=content,
                metadata={
                    "source": "RSA Constitution 1996",
                    "document_type": "legal",
                    "chapter": current_chapter
                }
            ))
    
    # 2. Update current chapter
    # We use match.start() so the chapter heading is INCLUDED in its own content
    current_chapter = match.group(0).strip()
    current_start = match.start()

# 3. Don't forget the last chapter (after the final match)
if current_start < len(text):
    content = text[current_start:].strip()
    if content:
        chapter_documents.append(Document(
            page_content=content,
            metadata={
                "source": "RSA Constitution 1996",
                "document_type": "legal",
                "chapter": current_chapter
            }
        ))

print(f"✓ Split into {len(chapter_documents)} chapter documents")


# --- 5. Configure the Text Splitter (Simplified to prevent micro-chunking) ---
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
    separators=[
        r"\n\n(?=\d+\.\s+[A-Z])",  # 1. Main sections (e.g., "9. Equality")
        "\n\n",                   # 2. Paragraph breaks
        " ",                      # 3. Spaces (prevents breaking mid-sentence)
        ""                        # 4. Characters (last resort)
    ],
    is_separator_regex=True,
)


# --- 6. Split and Filter ---
chunks = text_splitter.split_documents(chapter_documents)


# --- 7. Proof it works
print(f"\n✓ Total chunks: {len(chunks)}")

# Show a sample from different chapters
print("\n--- SAMPLE CHUNKS WITH METADATA ---")
seen_chapters = set()
for chunk in chunks:
    chapter = chunk.metadata['chapter']
    if chapter not in seen_chapters:
        seen_chapters.add(chapter)
        print(f"\nChapter: {chapter}")
        print(f"Preview: {chunk.page_content[:120]}...")
        if len(seen_chapters) >= 4:  # Show first 4 different chapters
            break

chunks[3]

✓ Removed Index starting at position: 306535

✓ Found 14 chapters
✓ Split into 15 chapter documents

✓ Total chunks: 403

--- SAMPLE CHUNKS WITH METADATA ---

Chapter: PREAMBLE
Preview: PREAMBLE We, the people of South Africa, Recognise the injustices of our past; Honour those who suffered for justice and...

Chapter: CHAPTER 1 FOUNDING PROVISIONS
Preview: CHAPTER 1 FOUNDING PROVISIONS Republic of South Africa 1. The Republic of South Africa is one, sovereign, democratic sta...

Chapter: CHAPTER 2 BILL OF RIGHTS
Preview: CHAPTER 2 BILL OF RIGHTS Rights 7. (1) This Bill of Rights is a cornerstone of democracy in South Africa. It enshrines t...

Chapter: CHAPTER 3 CO-OPERATIVE GOVERNMENT
Preview: CHAPTER 3 CO-OPERATIVE GOVERNMENT Government of the Republic 40. (1) In the Republic, government is constituted as natio...


Document(metadata={'source': 'RSA Constitution 1996', 'document_type': 'legal', 'chapter': 'CHAPTER 1 FOUNDING PROVISIONS'}, page_content='to the duties and responsibilities of citizenship. (3) National legislation must provide for the acquisition, loss and restoration of citizenship. National anthem 4. The national anthem of the Republic is determined by the President by proclamation. National flag 5. The national flag of the Republic is black, gold, green, white, red and blue, as described and sketched in Schedule 1.')